In [10]:
import pandas as pd
import numpy as np

train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

print(train.shape, test.shape)
train.head()

(94, 14) (41, 13)


,id,sexo,longitudCraneo,longitudPico,longitudNarina,anchoCraneo,altoPico,anchoPico,tarso,longAlaCerrada,longAlaAbierta,mediaEnvergadura,envergadura,peso
0,0J4_2016,H,174.18,106.85,80.19,52.43,30.20,23.26,88.74,51.0,95.7,104.2,208.4,2.38
1,A68_2016,H,171.12,104.59,79.45,51.14,30.61,24.04,90.22,50.5,94.4,102.2,204.4,2.14
2,E55_2015,H,174.69,106.83,82.14,49.09,30.60,26.47,91.42,52.7,96.6,105.2,210.4,2.50
3,7C7_2016,M,182.25,113.17,84.13,52.26,32.08,26.07,91.74,52.6,98.9,107.6,215.2,2.83
4,2C0_2018,H,173.70,105.19,80.37,49.20,34.00,27.12,87.77,50.8,94.5,103.7,207.4,2.73


In [11]:
# Check envergadura = 2 * mediaEnvergadura
check = train['envergadura'] - 2 * train['mediaEnvergadura']
print(check.describe())

count    94.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
dtype: float64


In [12]:
train.isna().sum()

,0
id,0
sexo,0
longitudCraneo,0
longitudPico,0
longitudNarina,0
anchoCraneo,0
altoPico,0
anchoPico,0
tarso,0
longAlaCerrada,0


In [13]:
test.isna().sum()

,0
id,0
longitudCraneo,0
longitudPico,0
longitudNarina,0
anchoCraneo,0
altoPico,0
anchoPico,0
tarso,0
longAlaCerrada,0
longAlaAbierta,0


In [14]:
train['peso'] = train.groupby('sexo')['peso'].transform(lambda x: x.fillna(x.median()))
train = train.drop(columns=['envergadura'])
test = test.drop(columns=['envergadura'])

In [15]:
features = ['longitudCraneo','longitudPico','longitudNarina','anchoCraneo','altoPico',
            'anchoPico','tarso','longAlaCerrada','longAlaAbierta','mediaEnvergadura','peso']

rows = []
for f in features:
    h = train[train.sexo == 'H'][f]
    m = train[train.sexo == 'M'][f]
    pooled_std = np.sqrt((h.std()**2 + m.std()**2) / 2)
    cohens_d = (m.mean() - h.mean()) / pooled_std
    rows.append({'feature': f, 'H_mean': round(h.mean(), 2), 'M_mean': round(m.mean(), 2),
                 'cohens_d': round(cohens_d, 2)})

effect_sizes = pd.DataFrame(rows).sort_values('cohens_d', key=abs, ascending=False)
effect_sizes

,feature,H_mean,M_mean,cohens_d
1,longitudPico,107.77,113.76,2.19
0,longitudCraneo,173.54,181.26,1.97
2,longitudNarina,81.33,85.41,1.70
10,peso,2.47,2.86,1.51
4,altoPico,31.89,33.52,1.48
6,tarso,89.68,92.75,1.29
3,anchoCraneo,51.77,54.10,1.24
9,mediaEnvergadura,103.94,106.04,0.94
8,longAlaAbierta,95.60,97.27,0.83
5,anchoPico,25.00,26.08,0.81


In [16]:
train['pico_craneo_ratio'] = train['longitudPico'] / train['longitudCraneo']
train['altoPico_anchoPico_ratio'] = train['altoPico'] / train['anchoPico']
train['tarso_craneo_ratio'] = train['tarso'] / train['longitudCraneo']
train['peso_ala_ratio'] = train['peso'] / train['longAlaCerrada']
train['narina_pico_ratio'] = train['longitudNarina'] / train['longitudPico']

test['pico_craneo_ratio'] = test['longitudPico'] / test['longitudCraneo']
test['altoPico_anchoPico_ratio'] = test['altoPico'] / test['anchoPico']
test['tarso_craneo_ratio'] = test['tarso'] / test['longitudCraneo']
test['peso_ala_ratio'] = test['peso'] / test['longAlaCerrada']
test['narina_pico_ratio'] = test['longitudNarina'] / test['longitudPico']

train.head()

,id,sexo,longitudCraneo,longitudPico,longitudNarina,anchoCraneo,altoPico,anchoPico,tarso,longAlaCerrada,longAlaAbierta,mediaEnvergadura,peso,pico_craneo_ratio,altoPico_anchoPico_ratio,tarso_craneo_ratio,peso_ala_ratio,narina_pico_ratio
0,0J4_2016,H,174.18,106.85,80.19,52.43,30.20,23.26,88.74,51.0,95.7,104.2,2.38,0.613446,1.298366,0.509473,0.046667,0.750491
1,A68_2016,H,171.12,104.59,79.45,51.14,30.61,24.04,90.22,50.5,94.4,102.2,2.14,0.611209,1.273295,0.527232,0.042376,0.759633
2,E55_2015,H,174.69,106.83,82.14,49.09,30.60,26.47,91.42,52.7,96.6,105.2,2.50,0.611540,1.156026,0.523327,0.047438,0.768885
3,7C7_2016,M,182.25,113.17,84.13,52.26,32.08,26.07,91.74,52.6,98.9,107.6,2.83,0.620960,1.230533,0.503374,0.053802,0.743395
4,2C0_2018,H,173.70,105.19,80.37,49.20,34.00,27.12,87.77,50.8,94.5,103.7,2.73,0.605584,1.253687,0.505296,0.053740,0.764046


In [19]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score

le = LabelEncoder()
y = le.fit_transform(train['sexo'])
x = train[features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_f1, rf_f1 = [], []
for train_idx, val_idx in cv.split(x, y):
    x_tr, x_val = x.iloc[train_idx], x.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    x_tr_scaled = scaler.fit_transform(x_tr)
    x_val_scaled = scaler.transform(x_val)

    log_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    log_model.fit(x_tr_scaled, y_tr)
    log_f1.append(f1_score(y_val, log_model.predict(x_val_scaled)))

    rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200)
    rf.fit(x_tr, y_tr)
    rf_f1.append(f1_score(y_val, rf.predict(x_val)))

print(f'Linear Regression F1: {np.mean(log_f1):.3f} | Random RandomForestClassifier F1: {np.mean(rf_f1):.3f}')

Linear Regression F1: 0.843 | Random RandomForestClassifier F1: 0.850


In [20]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(x_scaled, y)

coef_df = pd.DataFrame({'feature': features, 'coef': model.coef_[0]}).sort_values(
    'coef', key=abs, ascending=False)
coef_df

,feature,coef
4,altoPico,1.180220
10,peso,1.114268
1,longitudPico,1.044868
0,longitudCraneo,0.871157
6,tarso,0.675749
7,longAlaCerrada,-0.435738
9,mediaEnvergadura,0.313830
3,anchoCraneo,0.243724
8,longAlaAbierta,-0.212726
2,longitudNarina,0.111661


In [21]:
x_train = train[features]
x_test = test[features]

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

final_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
final_model.fit(x_train_scaled, y)

y_pred = final_model.predict(x_test_scaled)
y_pred_labels = le.inverse_transform(y_pred)

submission = pd.DataFrame({'id': test['id'], 'sexo': y_pred_labels})

submission.to_csv('submission_lr_tuned.csv', index=False)

print(submission['sexo'].value_counts())
submission.head(10)

sexo
H    21
M    20
Name: count, dtype: int64


,id,sexo
0,A70_2018,M
1,5C9_2018,H
2,5E1_2018,M
3,C58_2018,M
4,5E2_2018,M
5,5C2_2018,M
6,E21_2018,M
7,1C2_2018,M
8,C90_2018,M
9,A87_2018,M


In [22]:
rf_final = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200)
rf_final.fit(x_train, y)   # raw, unscaled -- trees don't need scaling

y_pred_rf = rf_final.predict(x_test)
y_pred_rf_labels = le.inverse_transform(y_pred_rf)

submission_rf = pd.DataFrame({'id': test['id'], 'sexo': y_pred_rf_labels})
submission_rf.to_csv('submission_rf_tuned.csv', index=False)

print(submission_rf['sexo'].value_counts())
submission_rf.head(10)

sexo
H    21
M    20
Name: count, dtype: int64


,id,sexo
0,A70_2018,M
1,5C9_2018,H
2,5E1_2018,M
3,C58_2018,M
4,5E2_2018,M
5,5C2_2018,M
6,E21_2018,M
7,1C2_2018,M
8,C90_2018,M
9,A87_2018,M
